In [ ]:
import cv2
import numpy as np
import glob
import os
from google.colab.patches import cv2_imshow

# ==============================
# 1. Checkerboard Configuration
# ==============================
CHECKERBOARD = (8, 6)   # inner corners (columns, rows)
SQUARE_SIZE = 24        # mm (or any consistent unit)

criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# ==============================
# 2. Prepare 3D Object Points
# ==============================
obj_3D = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
obj_3D[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
obj_3D *= SQUARE_SIZE

obj_points_3D = []
img_points_2D = []

# ==============================
# 3. Load Images
# ==============================
image_folder = "/content/*.JPG"
images = glob.glob(image_folder)

print(f"Found {len(images)} images")

image_size = None

# ==============================
# 4. Detect Corners
# ==============================
for fname in images:
    img = cv2.imread(fname)

    if img is None:
        print(f"Skipping unreadable file: {fname}")
        continue

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if image_size is None:
        image_size = gray.shape[::-1]  # (width, height)

    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

    if ret:
        obj_points_3D.append(obj_3D)

        corners2 = cv2.cornerSubPix(
            gray, corners, (11, 11), (-1, -1), criteria
        )
        img_points_2D.append(corners2)

        # Draw corners
        cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)

        # Show in Colab
        cv2_imshow(img)

    else:
        print(f"Chessboard NOT detected in {fname}")

# ==============================
# 5. Calibration
# ==============================
if len(obj_points_3D) == 0:
    raise ValueError("No valid checkerboard detections found.")

ret, mtx, dist_coeff, R_vecs, T_vecs = cv2.calibrateCamera(
    obj_points_3D,
    img_points_2D,
    image_size,
    None,
    None
)

print("\n✅ Calibration completed")
print(f"Reprojection Error: {ret}")

print("\nCamera Matrix:\n", mtx)
print("\nDistortion Coefficients:\n", dist_coeff)

# ==============================
# 6. Save Calibration Data
# ==============================
calib_data_path = "calibration_output"
os.makedirs(calib_data_path, exist_ok=True)

save_path = os.path.join(calib_data_path, "CalibrationMatrix.npz")

np.savez(
    save_path,
    camera_matrix=mtx,
    dist_coeff=dist_coeff,
    rvecs=R_vecs,
    tvecs=T_vecs,
    reprojection_error=ret
)

print(f"\n💾 Saved calibration to: {save_path}")

# ==============================
# 7. Undistortion Test
# ==============================
test_img = cv2.imread(images[0])
h, w = test_img.shape[:2]

new_camera_mtx, roi = cv2.getOptimalNewCameraMatrix(
    mtx, dist_coeff, (w, h), 1, (w, h)
)

undistorted = cv2.undistort(test_img, mtx, dist_coeff, None, new_camera_mtx)

# Crop result
x, y, w, h = roi
undistorted = undistorted[y:y+h, x:x+w]

print("\n📷 Original Image:")
cv2_imshow(test_img)

print("\n📷 Undistorted Image:")
cv2_imshow(undistorted)

# ==============================
# 8. Reprojection Error (Detailed)
# ==============================
total_error = 0

for i in range(len(obj_points_3D)):
    imgpoints2, _ = cv2.projectPoints(
        obj_points_3D[i], R_vecs[i], T_vecs[i], mtx, dist_coeff
    )
    error = cv2.norm(img_points_2D[i], imgpoints2, cv2.NORM_L2) / len(imgpoints2)
    total_error += error

mean_error = total_error / len(obj_points_3D)
print(f"\n📊 Mean Reprojection Error: {mean_error}")

In [ ]:
import numpy as np

data = np.load("calibration_output/CalibrationMatrix.npz")

print(data.files)  # shows all stored keys

['camera_matrix', 'dist_coeff', 'rvecs', 'tvecs', 'reprojection_error']


In [ ]:
camera_matrix = data["camera_matrix"]
dist_coeff = data["dist_coeff"]
rvecs = data["rvecs"]
tvecs = data["tvecs"]
error = data["reprojection_error"]

print("Camera Matrix:\n", camera_matrix)
print("Distortion:\n", dist_coeff)

Camera Matrix:
 [[3.17481733e+03 0.00000000e+00 1.54631655e+03]
 [0.00000000e+00 3.17659559e+03 2.01333116e+03]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion:
 [[ 1.88932518e-01 -1.47858012e+00 -3.64148317e-03  2.23494202e-03
   3.51652179e+00]]


In [ ]:
print("TVecs:\n", tvecs)
print("RVecs:\n", rvecs)

TVecs:
 [[[ -36.07839254]
  [-105.22124507]
  [ 362.71054546]]

 [[ -85.51494155]
  [ -86.42990279]
  [ 364.9397188 ]]

 [[-100.62333843]
  [ -46.0205715 ]
  [ 354.93590902]]

 [[ -92.51385155]
  [ -91.96734398]
  [ 343.52443468]]

 [[ -88.08498226]
  [ -89.30582874]
  [ 349.77993002]]

 [[ -72.19656194]
  [ -54.22999417]
  [ 323.3444538 ]]

 [[ -86.31607996]
  [ -64.99811786]
  [ 330.01214438]]

 [[ -49.60027754]
  [-141.97322781]
  [ 387.84661258]]]
RVecs:
 [[[ 9.47991401e-02]
  [-2.46599572e-02]
  [ 6.10780943e-01]]

 [[ 1.28652700e-01]
  [ 1.22903634e-01]
  [-1.98631799e-02]]

 [[-4.13149041e-04]
  [-3.32628888e-02]
  [-2.89484390e-01]]

 [[ 3.37023471e-01]
  [-2.15987316e-02]
  [ 2.36619515e-02]]

 [[ 1.11590026e-01]
  [-2.35828783e-01]
  [ 6.52022652e-03]]

 [[ 2.51573112e-03]
  [-2.37078493e-01]
  [-1.25906900e-02]]

 [[ 3.85272440e-01]
  [-4.69792228e-02]
  [ 1.77603441e-02]]

 [[ 2.93100195e-01]
  [-8.77891468e-02]
  [ 3.88679561e-01]]]


In [ ]:
import cv2
import plotly.express as px

# List to store the coordinates
corner_points = []

def get_coordinates(event, x, y, flags, param):
    global corner_points

    # If left mouse button is clicked, record the (u, v) coordinate
    if event == cv2.EVENT_LBUTTONDOWN:
        # u = x, v = y
        corner_points.append((x, y))
        print(f"Corner {len(corner_points)} recorded at (u={x}, v={y})")

        # Draw a small red circle where you clicked to confirm
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Click the 4 Corners", img_copy)

        # If we have 4 points, we are done
        if len(corner_points) == 4:
            print("\n--- Final Coordinates for this Image ---")
            for i, point in enumerate(corner_points):
                print(f"C{i+1}: u={point[0]}, v={point[1]}")
            print("Press any key to close the window.")

# --- Execution ---
if __name__ == "__main__":
    # 1. Put the name of your first photo here
    image_path = "/content/photos/origin.JPG"

    img = cv2.imread(image_path)

    if img is not None:
        # 2. OpenCV loads images in BGR color, but Plotly needs RGB. We must convert it.
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 3. Create an interactive plot
        fig = px.imshow(img_rgb, title="Hover over the object's corners to find (x, y) coordinates")
        fig.update_layout(width=1680, height=1400) # Significantly increase display size

        # 4. Display it right in the Colab notebook
        fig.show()
    else:
        print("Error: Could not load image. Did you upload it to the Colab files tab?")

In [ ]:
import cv2
import plotly.express as px

# List to store the coordinates
corner_points = []

def get_coordinates(event, x, y, flags, param):
    global corner_points

    # If left mouse button is clicked, record the (u, v) coordinate
    if event == cv2.EVENT_LBUTTONDOWN:
        # u = x, v = y
        corner_points.append((x, y))
        print(f"Corner {len(corner_points)} recorded at (u={x}, v={y})")

        # Draw a small red circle where you clicked to confirm
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Click the 4 Corners", img_copy)

        # If we have 4 points, we are done
        if len(corner_points) == 4:
            print("\n--- Final Coordinates for this Image ---")
            for i, point in enumerate(corner_points):
                print(f"C{i+1}: u={point[0]}, v={point[1]}")
            print("Press any key to close the window.")

# --- Execution ---
if __name__ == "__main__":
    # 1. Put the name of your first photo here
    image_path = "/content/photos/left.JPG"

    img = cv2.imread(image_path)

    if img is not None:
        # 2. OpenCV loads images in BGR color, but Plotly needs RGB. We must convert it.
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 3. Create an interactive plot
        fig = px.imshow(img_rgb, title="Hover over the object's corners to find (x, y) coordinates")
        fig.update_layout(width=1680, height=1400) # Significantly increase display size

        # 4. Display it right in the Colab notebook
        fig.show()
    else:
        print("Error: Could not load image. Did you upload it to the Colab files tab?")

In [ ]:
import cv2
import plotly.express as px

# List to store the coordinates
corner_points = []

def get_coordinates(event, x, y, flags, param):
    global corner_points

    # If left mouse button is clicked, record the (u, v) coordinate
    if event == cv2.EVENT_LBUTTONDOWN:
        # u = x, v = y
        corner_points.append((x, y))
        print(f"Corner {len(corner_points)} recorded at (u={x}, v={y})")

        # Draw a small red circle where you clicked to confirm
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Click the 4 Corners", img_copy)

        # If we have 4 points, we are done
        if len(corner_points) == 4:
            print("\n--- Final Coordinates for this Image ---")
            for i, point in enumerate(corner_points):
                print(f"C{i+1}: u={point[0]}, v={point[1]}")
            print("Press any key to close the window.")

# --- Execution ---
if __name__ == "__main__":
    # 1. Put the name of your first photo here
    image_path = "/content/photos/right.JPG"

    img = cv2.imread(image_path)

    if img is not None:
        # 2. OpenCV loads images in BGR color, but Plotly needs RGB. We must convert it.
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 3. Create an interactive plot
        fig = px.imshow(img_rgb, title="Hover over the object's corners to find (x, y) coordinates")
        fig.update_layout(width=1680, height=1400) # Significantly increase display size

        # 4. Display it right in the Colab notebook
        fig.show()
    else:
        print("Error: Could not load image. Did you upload it to the Colab files tab?")

In [ ]:
import cv2
import plotly.express as px

# List to store the coordinates
corner_points = []

def get_coordinates(event, x, y, flags, param):
    global corner_points

    # If left mouse button is clicked, record the (u, v) coordinate
    if event == cv2.EVENT_LBUTTONDOWN:
        # u = x, v = y
        corner_points.append((x, y))
        print(f"Corner {len(corner_points)} recorded at (u={x}, v={y})")

        # Draw a small red circle where you clicked to confirm
        cv2.circle(img_copy, (x, y), 5, (0, 0, 255), -1)
        cv2.imshow("Click the 4 Corners", img_copy)

        # If we have 4 points, we are done
        if len(corner_points) == 4:
            print("\n--- Final Coordinates for this Image ---")
            for i, point in enumerate(corner_points):
                print(f"C{i+1}: u={point[0]}, v={point[1]}")
            print("Press any key to close the window.")

# --- Execution ---
if __name__ == "__main__":
    # 1. Put the name of your first photo here
    image_path = "/content/photos/upright.JPG"

    img = cv2.imread(image_path)

    if img is not None:
        # 2. OpenCV loads images in BGR color, but Plotly needs RGB. We must convert it.
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # 3. Create an interactive plot
        fig = px.imshow(img_rgb, title="Hover over the object's corners to find (x, y) coordinates")
        fig.update_layout(width=1680, height=1400) # Significantly increase display size

        # 4. Display it right in the Colab notebook
        fig.show()
    else:
        print("Error: Could not load image. Did you upload it to the Colab files tab?")

In [6]:
import numpy as np

# 1. Camera Intrinsic Matrix
K = np.array([[3174.81733, 0.0, 1546.31655],
              [0.0, 3176.59559, 2013.33116],
              [0.0, 0.0, 1.0]])

# 2. Define the 4 Projection Matrices (M = K * [I | t])
# Origin: t = [0, 0, 0]
P1 = K @ np.array([[1, 0, 0, 0], [0, 1, 0, 0], [0, 0, 1, 0]])

# Right (1 unit): t = [-1, 0, 0]
P2 = K @ np.array([[1, 0, 0, -1], [0, 1, 0, 0], [0, 0, 1, 0]])

# Left (1 unit): t = [1, 0, 0]
P3 = K @ np.array([[1, 0, 0, 1], [0, 1, 0, 0], [0, 0, 1, 0]])

# Up-Right (1 unit right, 1 unit up): t = [-1, -1, 0]
P4 = K @ np.array([[1, 0, 0, -1], [0, 1, 0, 1], [0, 0, 1, 0]])

P_list = [P1, P2, P3, P4]

# 3. Your Extracted Pixel Coordinates
# Order: [Top-Left, Top-Right, Bottom-Right, Bottom-Left]
pts_cam1 = [[1369, 2327], [1697, 2327], [1697, 2552], [1369, 2552]]
pts_cam2 = [[1128, 2353], [1449, 2363], [1442, 2587], [1122, 2577]]
pts_cam3 = [[1604, 2314], [1938, 2317], [1935, 2542], [1604, 2535]]
pts_cam4 = [[1113, 2683], [1433, 2693], [1427, 2914], [1103, 2904]]

# 4. N-View Triangulation Function using SVD
def triangulate_nviews(P_matrices, points_2d):
    """
    Constructs the A matrix for DLT and solves AX = 0 using SVD.
    """
    A = []
    for P, (u, v) in zip(P_matrices, points_2d):
        # Two equations per view
        A.append(u * P[2, :] - P[0, :])
        A.append(v * P[2, :] - P[1, :])

    A = np.array(A)

    # SVD decomposes A into U, S, and V-transpose (Vt)
    _, _, Vt = np.linalg.svd(A)

    # The least-squares solution is the last row of Vt
    X = Vt[-1, :]

    # Convert from homogeneous (X, Y, Z, W) back to 3D (X, Y, Z)
    return X[:3] / X[3]

# 5. Calculate the 3D Coordinates for all 4 Corners
points_3D = []
print("--- 4-Camera SVD Reconstructed Coordinates (in Units) ---")
for i in range(4):
    # Gather the (u,v) for this specific corner across all 4 cameras
    corner_views = [pts_cam1[i], pts_cam2[i], pts_cam3[i], pts_cam4[i]]

    # Triangulate
    coord_3d = triangulate_nviews(P_list, corner_views)
    points_3D.append(coord_3d)

    print(f"Corner {i+1}: X={coord_3d[0]:.3f}, Y={coord_3d[1]:.3f}, Z={coord_3d[2]:.3f}")

# Convert list to numpy array for easy distance math
points_3D = np.array(points_3D)

# 6. Boundary Estimation (3D Distance Formula)
# Width = Distance between Top-Left (Index 0) and Top-Right (Index 1)
width_units = np.linalg.norm(points_3D[0] - points_3D[1])

# Height = Distance between Top-Left (Index 0) and Bottom-Left (Index 3)
height_units = np.linalg.norm(points_3D[0] - points_3D[3])

print("\n--- Final Multi-View Boundary Estimation ---")
print(f"Estimated Width:  {width_units * 4.5:.2f} inches")
print(f"Estimated Height: {height_units * 4.5:.2f} inches")

--- 4-Camera SVD Reconstructed Coordinates (in Units) ---
Corner 1: X=-0.664, Y=1.277, Z=11.945
Corner 2: X=0.555, Y=1.261, Z=11.656
Corner 3: X=0.539, Y=2.074, Z=11.617
Corner 4: X=-0.669, Y=2.089, Z=11.815

--- Final Multi-View Boundary Estimation ---
Estimated Width:  5.63 inches
Estimated Height: 3.70 inches
